In [1]:
import pandas as pd
import sqlite3

In [6]:
conn = sqlite3.connect("data/ledger.db")
# Fetch {name: id} pairs from categories table
ledger = pd.read_sql("SELECT * FROM ledger order by transaction_date DESC, amount ASC limit 100", conn)


In [7]:
ledger.shape

(100, 16)

In [9]:
ledger.head(10)

,id,transaction_date,description,amount,transaction_type,check_number,daily_posted_balance,running_balance,source_indicator,is_deleted,notes,more_note,category_id,fiscal_year_id,allocation_method_id,created_at
0,2635,pending,None,-4545.00,None,None,111378.10,111378.10,Import,0,None,None,9,10,None,2026-05-24 01:58:52
1,2633,pending,None,-1500.00,None,None,116304.85,116304.85,Import,0,None,None,19,10,None,2026-05-24 01:58:52
2,2634,pending,None,-381.75,None,None,115923.10,115923.10,Import,0,None,None,12,10,None,2026-05-24 01:58:52
3,2338,pending,None,0.00,Debit,None,78064.95,78064.95,Import,0,None,None,12,9,None,2026-05-24 01:58:52
4,2451,12/31/2025,TopScore YULA-VA MAIN ACCOUNT ACH CREDIT,96.56,Credit,None,85074.56,85074.56,Import,0,None,None,3,10,None,2026-05-24 01:58:52
5,1152,12/31/2021,TopScore YULA-VA MAIN ACCOUNT AC H CREDIT,55.98,Credit,None,42363.10,41963.10,Import,0,None,None,3,6,None,2026-05-24 01:58:52
6,889,12/31/2020,TopScore YULA-VA MAIN ACCOUNT AC H CREDIT,1094.88,Credit,None,27354.58,27019.08,Import,0,None,None,3,5,None,2026-05-24 01:58:52
7,692,12/31/2019,TRANSFER SPORTS WEB TOPSC MAIN A CCOUNT ACH CR...,37.62,Credit,None,35880.98,35880.98,Import,0,None,None,3,4,None,2026-05-24 01:58:52
8,2450,12/30/2025,TopScore YULA-VA MAIN ACCOUNT ACH CREDIT,183.67,Credit,None,84978.00,84978.00,Import,0,None,None,3,10,None,2026-05-24 01:58:52
9,1151,12/30/2021,TopScore YULA-VA MAIN ACCOUNT AC H CREDIT,18.66,Credit,None,42307.12,41907.12,Import,0,None,None,3,6,None,2026-05-24 01:58:52


In [8]:

# Fetch example script from site
script = pd.read_sql("""
            SELECT daily_posted_balance 
            FROM ledger 
            WHERE transaction_date != 'pending'
            ORDER BY transaction_date DESC, amount ASC 
            LIMIT 1
        """, conn)
script

,daily_posted_balance
0,85074.56


In [14]:

# Fetch example script from site
sample = pd.read_sql("""
            SELECT *
            FROM ledger l
            limit 10
        """, conn)
sample

,transaction_date,description,amount,transaction_type,check_number,daily_posted_balance,running_balance,category_id,fiscal_year_id,source_indicator,is_deleted,notes,more_notes
0,2017-09-01,Carry Forward,11143.93,credit,None,11143.93,11143.93,8,1,Import,0,Export from NPTreasurer,None
1,2017-09-01,170901P2 Square Inc 2963 David T,28.83,credit,None,11172.76,11172.76,6,2,Import,0,Export from NPTreasurer,None
2,2017-09-12,PAYMENT TO CREDIT CARD *********,-0.43,debit,None,11172.33,11172.33,15,2,Import,0,Export from NPTreasurer,None
3,2017-09-12,COUNTER DEPOSIT,607.00,credit,None,11779.33,11779.33,8,2,Import,0,Export from NPTreasurer,None
4,2017-09-13,RETURN DEPOSIT ITEM,-40.00,debit,None,11739.33,11739.33,15,2,Import,0,Export from NPTreasurer,None
5,2017-09-13,TRANSFER SPORTS WEB TOPSC X ACH,329.65,credit,None,12068.98,12068.98,3,2,Import,0,Export from NPTreasurer,None
6,2017-09-14,TRANSFER SPORTS WEB TOPSC X ACH,1065.31,credit,None,13134.29,13134.29,3,2,Import,0,Export from NPTreasurer,None
7,2017-09-15,TRANSFER SPORTS WEB TOPSC X ACH,164.70,credit,None,13298.99,13298.99,3,2,Import,0,Export from NPTreasurer,None
8,2017-09-18,TRANSFER SPORTS WEB TOPSC X ACH,147.34,credit,None,13446.33,13446.33,3,2,Import,0,Export from NPTreasurer,None
9,2017-09-19,TRANSFER SPORTS WEB TOPSC X ACH,305.10,credit,None,13751.43,13751.43,3,2,Import,0,Export from NPTreasurer,None


In [21]:

# Fetch example script from site
sample = pd.read_sql("""
            SELECT 
                p.name as program_name,
                c.flow as flow,
                SUM(l.amount * ifnull(ar.percentage,0)) as total
            FROM ledger l
            left join allocation_methods am on l.allocation_method_id = am.id
            left join allocation_rules ar on am.id = ar.method_id
            left JOIN programs p ON ar.program_id = p.id
            left JOIN categories c ON l.category_id = c.id
            WHERE l.fiscal_year_id = 7
            GROUP BY p.name, c.flow
            union all
            SELECT 
                'Total' as program_name,
                c.flow as flow,
                SUM(l.amount) as total
            FROM ledger l
            left JOIN categories c ON l.category_id = c.id
            WHERE l.fiscal_year_id = 7
            GROUP BY c.flow
            
        """, conn)
sample

,program_name,flow,total
0,None,Expense,0.00
1,None,Income,0.00
2,Total,Expense,-128821.93
3,Total,Income,122259.96


In [22]:

# Fetch example script from site
sample = pd.read_sql("""
            SELECT *
            FROM allocation_methods
            limit 10
        """, conn)
sample

,id,name,description
0,1,High School Fall,Direct entirely High School Fall
1,2,High School Winter,Direct entirely High School Winter
2,3,High School Spring,Direct entirely High School Spring
3,4,Middle School Fall,Direct entirely Middle School Fall
4,5,Middle School Spring,Direct entirely Middle School Spring
5,6,2026 Player Distribution,Split based on the distribution of players in ...
6,7,2026 Registration Distribution,Split based on the distribution of registratio...
7,8,2025 Player Distribution,Split based on the distribution of players in ...
8,9,2025 Registration Distribution,Split based on the distribution of registratio...
